# LSST SN Ia Simulation Pipeline

Forward-model SN Ia light curves with the Rubin/LSST 10-year baseline cadence using `lightcurvelynx`.

**Survey**: LSST baseline v3.4 OpSim (`baseline_v3.4_10yrs.db`)  
**Model**: SALT3 via `SncosmoWrapperModel`  
**Filters**: u, g, r, i, z, y  
**Redshift**: volumetric rate (Frohmaier et al. 2019), z = 0.01–1.2  
**Parameters**: Gaussian priors for x1 and c (no pzflow, no host galaxy)

## 1. Imports

## 0. Environment Setup

Set `LIGHTCURVELYNX_DATA_DIR` **before** importing lightcurvelynx — the download path is resolved at import time.  
Downloaded files (OpSim DB, passbands) will be stored in `./data/` inside this project directory.

In [ ]:
import os
from pathlib import Path

# Store downloaded data inside this project so it travels with the repo checkout.
# Must be set before any lightcurvelynx imports (path is resolved at import time).
_data_dir = Path().resolve() / "data"
_data_dir.mkdir(exist_ok=True)
os.environ["LIGHTCURVELYNX_DATA_DIR"] = str(_data_dir)
print(f"LIGHTCURVELYNX_DATA_DIR = {_data_dir}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from lightcurvelynx.obstable.opsim import OpSim
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.astro_utils.snia_utils import (
    DistModFromRedshift,
    X0FromDistMod,
    num_snia_per_redshift_bin,
    snia_volumetric_rates,
)
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.math_nodes.scipy_random import SamplePDF
from lightcurvelynx.math_nodes.ra_dec_sampler import ObsTableUniformRADECSampler
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel
from lightcurvelynx.simulate import simulate_lightcurves

## 2. Simulation Configuration

In [ ]:
SEED = 1024
RNG  = np.random.default_rng(SEED)

SIM_PARAMS = {
    # Cosmology
    "H0": 73.0,
    "Omega_m": 0.3,
    # Redshift range
    "zmin": 0.01,
    "zmax": 1.2,
    "znbins": 100,
    # Tripp relation coefficients
    "alpha": 0.14,
    "beta": 3.1,
    # SALT3 Gaussian priors (no pzflow, no host galaxy)
    "x1_mean": 0.0,  "x1_sigma": 1.0,
    "c_mean":  0.0,  "c_sigma":  0.1,
    "m_abs_mean": -19.3, "m_abs_sigma": 0.12,
    # Survey
    "filters": ["u", "g", "r", "i", "z", "y"],
    "sky_coverage": 18_000.0,  # LSST WFD footprint in deg²
}

## 3. Load LSST OpSim

The file is downloaded once and cached locally by `OpSim.from_url()`.

In [ ]:
OPSIM_URL = (
    "https://s3df.slac.stanford.edu/data/rubin/sim-data/"
    "sims_featureScheduler_runs5.0/baseline/baseline_v5.0.1_10yrs.db"
)
opsim = OpSim.from_url(OPSIM_URL)
print(f"OpSim loaded: {len(opsim):,} observations")
print(f"MJD range: {opsim['time'].min():.1f} – {opsim['time'].max():.1f}")
print(f"Filters:    {sorted(opsim['filter'].unique())}")

## 4. Calculate Number of SNe to Simulate

Integrate the volumetric SN Ia rate over the survey volume and duration:
$$N = \int_{z_{\rm min}}^{z_{\rm max}} r_v(z)\,\frac{dV}{dz}\,\frac{dz}{1+z} \times \Omega \times T_{\rm survey}$$

In [ ]:
t_min = float(opsim["time"].min())
t_max = float(opsim["time"].max())

survey_length = (t_max - t_min) / 365.25
print(f"Survey length = {survey_length:.2f} years")

solid_angle = SIM_PARAMS["sky_coverage"] * (np.pi / 180.0) ** 2
print(f"Solid angle   = {solid_angle:.4f} sr  ({SIM_PARAMS['sky_coverage']:,.0f} deg²)")

nsntotal, _ = num_snia_per_redshift_bin(
    SIM_PARAMS["zmin"],
    SIM_PARAMS["zmax"],
    znbins=1,
    solid_angle=solid_angle,
    vol_rate_function=snia_volumetric_rates,
    H0=SIM_PARAMS["H0"],
    Omega_m=SIM_PARAMS["Omega_m"],
)
nsn = int(nsntotal[0] * survey_length)
print(f"Expected SNe Ia = {nsn:,}")

## 5. Load LSST Passbands

In [ ]:
passbands = PassbandGroup.from_preset("LSST", filters=SIM_PARAMS["filters"])
print(passbands)

## 6. Redshift Distribution (Volumetric Rate)

Use the Frohmaier et al. (2019) volumetric rate $r_v(z) = r_0\,(1+z)^\alpha$ to compute the
expected number of SNe Ia per redshift bin, then build an interpolated PDF for sampling.

In [ ]:
nsn_per_bin, z_mean = num_snia_per_redshift_bin(
    SIM_PARAMS["zmin"],
    SIM_PARAMS["zmax"],
    SIM_PARAMS["znbins"],
    H0=SIM_PARAMS["H0"],
    Omega_m=SIM_PARAMS["Omega_m"],
)
zpdf = interp1d(z_mean, nsn_per_bin, bounds_error=False, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 4))
dz = (SIM_PARAMS["zmax"] - SIM_PARAMS["zmin"]) / SIM_PARAMS["znbins"]
ax.bar(z_mean, nsn_per_bin, width=dz, color="steelblue", alpha=0.8)
ax.set(xlabel="Redshift", ylabel="SN Ia yr$^{-1}$ per bin",
       title="Volumetric rate redshift distribution")
plt.tight_layout()
plt.show()

## 7. Build SN Ia Source Model

Parameter graph:
- **RA, Dec** — uniformly sampled from the observed LSST footprint (`ObsTableUniformRADECSampler`)
- **redshift** — drawn from the volumetric rate PDF (`SamplePDF`)
- **x1** — Gaussian($\mu=0$, $\sigma=1$)
- **c** — Gaussian($\mu=0$, $\sigma=0.1$)
- **M_abs** — Gaussian($\mu=-19.3$, $\sigma=0.12$)
- **distmod** — computed from redshift via `DistModFromRedshift`
- **x0** — computed via the Tripp relation through `X0FromDistMod`

In [ ]:
# RA/Dec: uniform over the opsim footprint (rejection sampling)
radec = ObsTableUniformRADECSampler(opsim, node_label="radec")

# Redshift from volumetric rate PDF
z_func = SamplePDF(zpdf, node_label="redshift")

# Gaussian SALT3 parameter priors
x1_func    = NumpyRandomFunc("normal", loc=SIM_PARAMS["x1_mean"],    scale=SIM_PARAMS["x1_sigma"])
c_func     = NumpyRandomFunc("normal", loc=SIM_PARAMS["c_mean"],     scale=SIM_PARAMS["c_sigma"])
m_abs_func = NumpyRandomFunc("normal", loc=SIM_PARAMS["m_abs_mean"], scale=SIM_PARAMS["m_abs_sigma"])

# x0 via Tripp relation
distmod_func = DistModFromRedshift(
    z_func, H0=SIM_PARAMS["H0"], Omega_m=SIM_PARAMS["Omega_m"]
)
x0_func = X0FromDistMod(
    distmod=distmod_func,
    x1=x1_func,
    c=c_func,
    alpha=SIM_PARAMS["alpha"],
    beta=SIM_PARAMS["beta"],
    m_abs=m_abs_func,
    node_label="x0_func",
)

# Assemble the SALT3 source (no host galaxy)
source = SncosmoWrapperModel(
    "salt3",
    t0=NumpyRandomFunc("uniform", low=t_min, high=t_max),
    x0=x0_func,
    x1=x1_func,
    c=c_func,
    ra=radec.ra,
    dec=radec.dec,
    redshift=z_func,
    node_label="source",
)
print("Source model built successfully.")

## 8. Run Simulation

In [ ]:
param_cols = [
    "source.t0",
    "source.x0",
    "source.x1",
    "source.c",
    "source.redshift",
    "source.ra",
    "source.dec",
    "x0_func.distmod",
]
obstable_save_cols = ["filter", "time", "skyBrightness", "seeingFwhmEff", "fiveSigmaDepth"]

results = simulate_lightcurves(
    model=source,
    num_samples=nsn,
    obstable=opsim,
    passbands=passbands,
    param_cols=param_cols,
    obstable_save_cols=obstable_save_cols,
    rest_time_window_offset=(-20, 50),  # rest-frame phase window in days
    rng=RNG,
    num_jobs=8,
    batch_size=5000,
)
print(f"Simulated {len(results):,} SNe Ia")
results.head()

## 9. Diagnostics

Quick sanity checks on the simulated population.

In [ ]:
# TODO: redshift distribution of simulated SNe
# fig, ax = plt.subplots()
# results["source.redshift"].hist(bins=50, ax=ax)
# ax.set(xlabel="Redshift", ylabel="Count", title="Simulated redshift distribution")
# plt.show()

In [ ]:
# TODO: x1 and c distributions
# fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# results["source.x1"].hist(bins=50, ax=axes[0])
# axes[0].set(xlabel="x1", ylabel="Count")
# results["source.c"].hist(bins=50, ax=axes[1])
# axes[1].set(xlabel="c", ylabel="Count")
# plt.tight_layout()
# plt.show()

In [ ]:
# TODO: Hubble diagram (distmod vs redshift)
# fig, ax = plt.subplots()
# ax.scatter(results["source.redshift"], results["x0_func.distmod"], s=2, alpha=0.3)
# ax.set(xlabel="Redshift", ylabel="Distance modulus", title="Hubble diagram")
# plt.show()

In [ ]:
# TODO: example light curve for a single SN
# sn = results.iloc[0]
# lc = sn["lightcurve"]
# for band in SIM_PARAMS["filters"]:
#     mask = lc["filter"] == f"LSST_{band}"
#     if mask.any():
#         plt.errorbar(lc["time"][mask], lc["flux"][mask], lc["flux_err"][mask],
#                      fmt="o", label=band, capsize=3)
# plt.axvline(sn["source.t0"], ls="--", color="k", label="t0")
# plt.legend()
# plt.xlabel("MJD")
# plt.ylabel("Flux (nJy)")
# plt.title(f'Example SN Ia  z={sn["source.redshift"]:.3f}')
# plt.show()

## 10. Save Results

*(Placeholder — fill in output path as needed.)*

In [ ]:
# TODO: save results
# output_path = "lsst_snia_results.parquet"
# results.to_parquet(output_path)
# print(f"Saved to {output_path}")